In [ ]:
import json
from datetime import date
from pathlib import Path

import polars as pl
import yaml

from common import read_day, daterange
from clean import clean_os_version, merge_partitions

with open("../config.yaml", "r") as fp:
    config = yaml.safe_load(fp)

days = []
for i in config["days"]:
    days += list(daterange(i["start"], i["end"]))

In [ ]:
OUTPUT = f"{config['dirs']['output']}/{config['city']}"
SET_TIMEZONE = True

In [ ]:
with open(f"{OUTPUT}/id_lookup/id_lookup.json", "r") as fp:
    lookup = json.load(fp)

with open("../data/os_version.yaml", "r") as fp:
    os_versions = yaml.safe_load(fp)
os_versions["ios"] = {k: v.strftime("%Y-%m-%d") for k, v in os_versions["ios"].items()}
os_versions["android"] = {
    k: v.strftime("%Y-%m-%d") if isinstance(v, date) else v
    for k, v in os_versions["android"].items()
}

In [ ]:
for i in days:
    data = read_day(i, config["dirs"]["raw_source"])
    data = data.with_columns(pl.from_epoch(pl.col("timestamp"), time_unit="s"))

    # Clean OS version
    if "OS_version" not in config["columns_to_drop"]:
        data = data.with_columns(
            pl.col("OS_version")
            .fill_null("")
            .map_elements(clean_os_version, return_dtype=float)
        )

    # Drop insignificant columns
    data = data.drop(config["columns_to_drop"])

    # Set timezone
    dt_format = "%Y-%m-%d %H:%M:%S"
    if SET_TIMEZONE:
        data = data.with_columns(
            timestamp=pl.col("timestamp").dt.convert_time_zone(
                time_zone="Europe/Budapest"
            ),
        )
        dt_format = "%Y-%m-%d %H:%M:%S %z"

    data = data.with_columns(
        pl.col("device_aid_hashed").str.slice(0, 2).alias("partition")
    )
    # Replace device IDs
    data = data.with_columns(pl.col("device_aid_hashed").replace_strict(lookup))
    data = data.rename({"device_aid_hashed": "device_id"})

    for p in data.partition_by("partition"):
        path = f"{OUTPUT}/partitioned/{p['partition'][0]}"
        Path(path).mkdir(exist_ok=True, parents=True)
        p_ = p.sort(["device_id", "timestamp"]).drop("partition")
        p_.write_parquet(f"{path}/pings_{i.strftime('%Y_%m_%d')}.parquet")

        del p_
    del data

    # data.sort("timestamp").write_csv(f"{path}/locations.csv", datetime_format=dt_format)

## merge partitions

In [ ]:
missing = merge_partitions(f"{OUTPUT}/partitioned")

Path(f"{OUTPUT}/debug/").mkdir(exist_ok=True, parents=True)
with open(f"{OUTPUT}/debug/03_merge_partitions_missing_log.json", "w") as fp:
    json.dump([(i[0], i[1].strftime("%Y-%m-%d")) for i in missing], fp, indent=1)

# runs 45m for Budapest data

## save partition 00 as CSV for debugging

In [ ]:
df = pl.read_parquet(f"{OUTPUT}/partitioned/00/pings.parquet")
df.write_csv(f"{OUTPUT}/partitioned/00/pings.csv")